# Очищення та аналіз текстів із користувацьких відгуків

## Бізнес-контекст
Компанія Feedback Insight отримує сотні відгуків клієнтів про свої продукти щодня. Вам необхідно автоматизувати процес аналізу текстів: очистити дані, виконати токенізацію, видалити шум і виявити найпоширеніші слова і фрази, що відображають думки клієнтів.

## Завдання 1
Підготуйте та завантажте дані.
Збережіть у CSV-файл 10-15 текстових відгуків клієнтів (англійською мовою).
Завантажте дані за допомогою pandas.
Обробіть тексти: приведіть до нижнього регістру, видаліть спецсимволи та цифри.
Перевірте кількість рядків і середню довжину відгуків (у словах).

In [ ]:
import pandas as pd
import re

reviews = [
    "Amazing product! Totally loved it.",
    "Very bad quality, will not buy 123 again.",
    "The best thing I have ever bought!!!",
    "Average experience. Not too bad but not great.",
    "Such terrible packaging and slow delivery.",
    "Excellent customer service and fantastic product.",
    "I am very disappointed with this item.",
    "Highly recommended to everyone!",
    "It broke after 2 days of use. So sad.",
    "Absolutely wonderful, 10/10 would buy again.",
    "Poor materials used. Feels very cheap.",
    "Great value for money.",
    "I hate it. Totally useless object.",
    "Works perfectly, just as described.",
    "Not worth the price at all."
]

df = pd.DataFrame({"review": reviews})
df.to_csv("customer_reviews.csv", index=False)

df = pd.read_csv("customer_reviews.csv")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['cleaned'] = df['review'].apply(clean_text)

num_rows = len(df)
avg_length = df['cleaned'].apply(lambda x: len(x.split())).mean()

print(num_rows)
print(avg_length)

## Завдання 2
Застосуйте токенізацію і видаліть стоп-слова.
Розділіть тексти на токени за допомогою nltk.word_tokenize.
Видаліть стоп-слова з бібліотеки nltk.corpus.stopwords.
Видаліть усі токени довжиною менше 3 символів.
Підрахуйте загальну кількість токенів до і після очищення.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

total_tokens_before = 0
total_tokens_after = 0
cleaned_tokens_list = []

for text in df['cleaned']:
    tokens = word_tokenize(text)
    total_tokens_before += len(tokens)
    
    filtered_tokens = [w for w in tokens if w not in stop_words and len(w) >= 3]
    total_tokens_after += len(filtered_tokens)
    
    cleaned_tokens_list.append(filtered_tokens)

df['tokens'] = cleaned_tokens_list

print(total_tokens_before)
print(total_tokens_after)

## Завдання 3
Побудуйте частотний словник і візуалізуйте результати.
Використовуйте collections.Counter для підрахунку зустрічальності слів.
Виведіть 15 найчастіших слів та їхню кількість.
Побудуйте горизонтальну діаграму за допомогою matplotlib:
Підпишіть осі та заголовок графіка (Top 15 Frequent Words).
Збережіть зображення як feedback_word_freq.png.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

all_tokens = [token for tokens_list in df['tokens'] for token in tokens_list]
word_counts = Counter(all_tokens)

top_15 = word_counts.most_common(15)
for word, freq in top_15:
    print(word, freq)

words = [item[0] for item in top_15]
freqs = [item[1] for item in top_15]

plt.barh(words[::-1], freqs[::-1], color='skyblue')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.title('Top 15 Frequent Words')
plt.tight_layout()
plt.savefig('feedback_word_freq.png')
plt.show()

## Завдання 4
Додатковий аналіз біграм.
Згенеруйте біграми (послідовності з двох слів) за допомогою nltk.bigrams.
Підрахуйте частоту появи біграм.
Відобразіть 10 найчастіших біграм у вигляді таблиці.
Збережіть результати в CSV-файл feedback_bigrams.csv.

In [ ]:
from nltk import bigrams

all_bigrams = list(bigrams(all_tokens))
bigram_counts = Counter(all_bigrams)

top_10_bigrams = bigram_counts.most_common(10)

bigram_strings = [f"{b[0]} {b[1]}" for b, f in top_10_bigrams]
bigram_freqs = [f for b, f in top_10_bigrams]

df_bigrams = pd.DataFrame({
    'Біграма': bigram_strings,
    'Частота': bigram_freqs
})

display(df_bigrams)

df_bigrams.to_csv('feedback_bigrams.csv', index=False)